# Assignment 2

In this assigment, we will work with the *Adult* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/2/adult). Extract the data files into the subdirectory: `../05_src/data/adult/` (relative to `./05_src/`).

# Load the data

Assuming that the files `adult.data` and `adult.test` are in `../05_src/data/adult/`, then you can use the code below to load them.

In [15]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, balanced_accuracy_score
import logging
import warnings

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO)

columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country', 'income'
]
adult_dt = (pd.read_csv('../../05_src/data/adult/adult.data', header = None, names = columns)
              .assign(income = lambda x: (x.income.str.strip() == '>50K')*1))

# Get X and Y

Create the features data frame and target data:

+ Create a dataframe `X` that holds the features (all columns that are not `income`).
+ Create a dataframe `Y` that holds the target data (`income`).
+ From `X` and `Y`, obtain the training and testing data sets:

    - Use a train-test split of 70-30%. 
    - Set the random state of the splitting function to 42.

In [16]:
# Create features (X) and target (Y)
X = adult_dt.drop(columns=['income'])
Y = adult_dt[['income']]

# Perform train-test split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

logging.info(f"Train-test split done: {X_train.shape}, {X_test.shape}, {Y_train.shape}, {Y_test.shape}")


2024-07-06 09:15:38,903 - INFO - Train-test split done: (22792, 14), (9769, 14), (22792, 1), (9769, 1)


## Random States

Please comment: 

+ What is the [random state](https://scikit-learn.org/stable/glossary.html#term-random_state) of the [splitting function](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)? 
+ Why is it [useful](https://en.wikipedia.org/wiki/Reproducibility)?

- The random state in the splitting function is like a specific starting point for the random number generator. It ensures that the way the data is divided into training and testing parts happens in the same way every time you run the code. In our case, the random state is set to 42.

- The random state is useful because it makes the results consistent and repeatable. When someone else runs the code again with the same random state, the data will split in the exact same way. This helps in verifying results and makes it easier to compare different experiments or improvements since everyone will be working with the same data splits. This consistency is important in scientific and technical work to ensure that findings can be checked and confirmed by others.

# Preprocessing

Create a [Column Transformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) that treats the features as follows:

- Numerical variables

    * Apply [KNN-based imputation for completing missing values](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html):
        
        + Consider the 7 nearest neighbours.
        + Weight each neighbour by the inverse of its distance, causing closer neigbours to have more influence than more distant ones.
    * [Scale features using statistics that are robust to outliers](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html#sklearn.preprocessing.RobustScaler).

- Categorical variables: 
    
    * Apply a [simple imputation strategy](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html#sklearn.impute.SimpleImputer):

        + Use the most frequent value to complete missing values, also called the *mode*.

    * Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html):
        
        + Handle unknown labels if they exist.
        + Drop one column for binary variables.
    
    
The column transformer should look like this:

![](./images/assignment_2__column_transformer.png)

In [17]:
# Define numerical and categorical pipelines
num_pipeline = Pipeline([
    ('imputer', KNNImputer(n_neighbors=7, weights='distance')),
    ('scaler', RobustScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='if_binary'))
])

# Create the ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num_transforms', num_pipeline, X.select_dtypes(include=['int64', 'float64']).columns),
        ('cat_transforms', cat_pipeline, X.select_dtypes(include=['object']).columns)
    ]
)

logging.info("Preprocessor configured.")
preprocessor


2024-07-06 09:15:38,944 - INFO - Preprocessor configured.


ColumnTransformer(transformers=[('num_transforms',
                                 Pipeline(steps=[('imputer',
                                                  KNNImputer(n_neighbors=7,
                                                             weights='distance')),
                                                 ('scaler', RobustScaler())]),
                                 Index(['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss',
       'hours-per-week'],
      dtype='object')),
                                ('cat_transforms',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='if_binary',
                                                                handle_unknown='ignore'))]),
                                 Index(['workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'race', 'sex', 'native-country'],
      dtype='object'))])

## Model Pipeline

Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `classifier` and assign a [`RandomForestClassifier()`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) to it.

The pipeline looks like this:

![](./images/assignment_2__pipeline.png)

In [18]:
# Create the model pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier())
])

# Display the model pipeline configuration

logging.info("Model pipeline configured.")
model_pipeline


2024-07-06 09:15:39,050 - INFO - Model pipeline configured.


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num_transforms',
                                                  Pipeline(steps=[('imputer',
                                                                   KNNImputer(n_neighbors=7,
                                                                              weights='distance')),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  Index(['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss',
       'hours-per-week'],
      dtype='object')),
                                                 ('cat_transforms',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(drop='if_binary',
                                                                                 handle_unknown='ignore'))]),
                                                  Index(['workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'race', 'sex', 'native-country'],
      dtype='object'))])),
                ('classifier', RandomForestClassifier())])

# Cross-Validation

Evaluate the model pipeline using [`cross_validate()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html):

+ Measure the following [performance metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#common-cases-predefined-values): negative log loss, ROC AUC, accuracy, and balanced accuracy.
+ Report the training and validation results. 
+ Use five folds.


In [19]:
# Perform cross-validation
scoring = ['neg_log_loss', 'roc_auc', 'accuracy', 'balanced_accuracy']
cv_results = cross_validate(model_pipeline, X_train, Y_train, cv=5, scoring=scoring, return_train_score=True)
cv_results_df = pd.DataFrame(cv_results)

# Display the cross-validation results
print("Cross-Validation Results:")
logging.info("Cross-validation completed.")
print(cv_results_df)


2024-07-06 09:16:40,887 - INFO - Cross-validation completed.


Cross-Validation Results:
    fit_time  score_time  test_neg_log_loss  train_neg_log_loss  test_roc_auc  \
0  11.324457    0.216791          -0.340253           -0.081495      0.905690   
1  11.070263    0.200174          -0.380554           -0.081108      0.902243   
2  10.986635    0.200131          -0.382387           -0.081684      0.901463   
3  11.687395    0.270910          -0.357193           -0.081947      0.907010   
4  11.899646    0.233447          -0.381246           -0.081584      0.901716   

   train_roc_auc  test_accuracy  train_accuracy  test_balanced_accuracy  \
0            1.0       0.852161        1.000000                0.777342   
1            1.0       0.849748        1.000000                0.769910   
2            1.0       0.849057        1.000000                0.767763   
3            1.0       0.860465        0.999945                0.782985   
4            1.0       0.856955        1.000000                0.776361   

   train_balanced_accuracy  
0      

Display the fold-level results as a pandas data frame and sorted by negative log loss of the test (validation) set.

In [20]:
# Sort and display the cross-validation results by test negative log loss
cv_results_df_sorted = cv_results_df.sort_values(by='test_neg_log_loss')
print("Sorted Cross-Validation Results:")
print(cv_results_df_sorted)

Sorted Cross-Validation Results:
    fit_time  score_time  test_neg_log_loss  train_neg_log_loss  test_roc_auc  \
2  10.986635    0.200131          -0.382387           -0.081684      0.901463   
4  11.899646    0.233447          -0.381246           -0.081584      0.901716   
1  11.070263    0.200174          -0.380554           -0.081108      0.902243   
3  11.687395    0.270910          -0.357193           -0.081947      0.907010   
0  11.324457    0.216791          -0.340253           -0.081495      0.905690   

   train_roc_auc  test_accuracy  train_accuracy  test_balanced_accuracy  \
2            1.0       0.849057        1.000000                0.767763   
4            1.0       0.856955        1.000000                0.776361   
1            1.0       0.849748        1.000000                0.769910   
3            1.0       0.860465        0.999945                0.782985   
0            1.0       0.852161        1.000000                0.777342   

   train_balanced_accuracy  


Calculate the mean of each metric. 

In [21]:
# Calculate the mean of each metric
cv_results_mean = cv_results_df.mean()
print("The mean of each metric")
print(cv_results_mean)


The mean of each metric
fit_time                   11.393679
score_time                  0.224291
test_neg_log_loss          -0.368327
train_neg_log_loss         -0.081564
test_roc_auc                0.903625
train_roc_auc               1.000000
test_accuracy               0.853677
train_accuracy              0.999989
test_balanced_accuracy      0.774872
train_balanced_accuracy     0.999977
dtype: float64


Calculate the same performance metrics (negative log loss, ROC AUC, accuracy, and balanced accuracy) using the testing data `X_test` and `Y_test`. Display results as a dictionary.

*Tip*: both, `roc_auc()` and `neg_log_loss()` will require prediction scores from `pipe.predict_proba()`. However, for `roc_auc()` you should only pass the last column `Y_pred_proba[:, 1]`. Use `Y_pred_proba` with `neg_log_loss()`.

In [22]:
# Fit the model pipeline on the training data
model_pipeline.fit(X_train, Y_train)

# Predict probabilities on the testing data
Y_pred_proba = model_pipeline.predict_proba(X_test)

# Calculate the performance metrics
results = {
    'Negative Log Loss': log_loss(Y_test, Y_pred_proba),
    'ROC AUC': roc_auc_score(Y_test, Y_pred_proba[:, 1]),
    'Accuracy': accuracy_score(Y_test, model_pipeline.predict(X_test)),
    'Balanced Accuracy': balanced_accuracy_score(Y_test, model_pipeline.predict(X_test))
}

# Display the results
print("Performance Metrics on Testing Data:")
print(results)

Performance Metrics on Testing Data:
{'Negative Log Loss': 0.4032054059703039, 'ROC AUC': 0.8997218111318444, 'Accuracy': 0.8533114955471389, 'Balanced Accuracy': 0.7720188315140049}


# Target Recoding

In the first code chunk of this document, we loaded the data and immediately recoded the target variable `income`. Why is this [convenient](https://scikit-learn.org/stable/modules/model_evaluation.html#binary-case)?

The specific line was:

```
adult_dt = (pd.read_csv('../05_src/data/adult/adult.data', header = None, names = columns)
              .assign(income = lambda x: (x.income.str.strip() == '>50K')*1))
```

### Why Target Recoding is Convenient

Recoding the target variable `income` right after loading the data simplifies the workflow and reduces errors. Converting the income column to a binary format (0 or 1) makes the data easier to work with, as many machine learning models and evaluation metrics in sci-kit-learn expect binary targets. Handling the target variable at the start prevents mistakes later. It ensures seamless processing in subsequent steps like splitting the data, training the model, and evaluating performance. This early recoding step also ensures compatibility with many algorithms and evaluation metrics for binary values.

## Criteria

|Criteria|Complete|Incomplete|
|---------------------|----|----|
|Evaluation of model pipeline |Model pipeline was evaluated correctly.|Model pipeline was not evaluated correctly.|
|Explanation of answer|Answer was concise and explained the learner's reasoning in depth.|Answer was not concise and did not explained the learner's reasoning in depth.|

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `11:59 PM - 05/07/2024`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/fredylrincon/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [X] Created a branch with the correct naming convention.
- [X] Ensured that the repository is public.
- [X] Reviewed the PR description guidelines and adhered to them.
- [X] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Becker,Barry and Kohavi,Ronny. (1996). Adult. UCI Machine Learning Repository. https://doi.org/10.24432/C5XW20.